# Load Address Prediction Results

This notebook reads `results.csv` and plots runtime vs. iterations for each access-pattern and core-class combination.

In [1]:
import altair as alt
import duckdb

In [2]:
con = duckdb.connect()

df = con.sql("""
    SELECT
        iters AS iterations,
        pattern,
        core_kind,
        min,
        p25,
        median,
        p75,
        max,
        first,
        last,
        unit
    FROM read_csv_auto('results.csv')
    ORDER BY core_kind, pattern, iterations
""").df()

df.head()

,iterations,pattern,core_kind,min,p25,median,p75,max,first,last,unit
0,10,random,ecore,4343,4442,4474,4503,34597,7605,4388,cycles
1,20,random,ecore,4419,4495,4542,4611,26236,5526,4515,cycles
2,30,random,ecore,4510,4599,4637,4690,37475,5026,4629,cycles
3,40,random,ecore,4597,4690,4746,4794,52115,5200,4644,cycles
4,50,random,ecore,4689,4797,4832,4876,44846,5695,4788,cycles


In [ ]:
unit = df["unit"].iloc[0]
y_title = f"Runtime ({unit})"

base = (
    alt.Chart(df)
    .mark_line(point=True)
    .encode(
        x=alt.X("iterations:Q", title="Iterations"),
        y=alt.Y("median:Q", title=y_title),
        color=alt.Color("pattern:N", title="Pattern"),
        tooltip=[
            "pattern", "core_kind", "iterations",
            "min", "p25", "median", "p75", "max",
            "first", "last", "unit",
        ],
    )
    .properties(width=380, height=280)
)

chart = base.facet(
    column=alt.Column("core_kind:N", title="Core Class", sort=["ecore", "pcore"])
).properties(title="Runtime vs. Iterations by Core Class").interactive()

chart